# 04 SHAP
Explain the tuned Random Forest model globally and per student.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

model = joblib.load('../src/model.pkl')
feature_cols = joblib.load('../src/feature_cols.pkl')
engine = create_engine('sqlite:///../data/students.db')
df = pd.read_sql(text('SELECT * FROM students'), engine)
X = df.drop(columns=['G1','G2','G3','pass']).select_dtypes(include=[np.number])[feature_cols]
y = df['pass']

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)
def class_one_values(values):
    if isinstance(values, list):
        return values[1]
    arr = np.asarray(values)
    return arr[:, :, 1] if arr.ndim == 3 else arr
class_one = class_one_values(shap_values)

In [ ]:
shap.summary_plot(class_one, X, plot_type='bar', max_display=15, show=False)
plt.tight_layout()
plt.savefig('../data/shap_global_importance.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
shap.summary_plot(class_one, X, plot_type='dot', max_display=15, show=False)
plt.tight_layout()
plt.savefig('../data/shap_dot_plot.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
high_risk_idx = X.loc[y == 0, 'risk_index'].idxmax()
expected_value = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
shap.force_plot(expected_value, class_one[X.index.get_loc(high_risk_idx), :], X.loc[high_risk_idx, :], matplotlib=True, show=False)
plt.savefig('../data/shap_force_plot.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
def get_top_reasons(student_row, n=3):
    row = student_row.to_frame().T if isinstance(student_row, pd.Series) else student_row
    values = class_one_values(explainer.shap_values(row))[0]
    reasons = []
    for idx in np.argsort(np.abs(values))[::-1][:n]:
        feature = row.columns[idx]
        value = row.iloc[0, idx]
        shap_value = values[idx]
        direction = 'reduces' if shap_value > 0 else 'increases'
        reasons.append(f'{feature} = {value} {direction} failure risk (impact: {shap_value:+.3f})')
    return reasons

for idx in X.sample(5, random_state=42).index:
    print(idx, get_top_reasons(X.loc[idx], n=3))

In [ ]:
joblib.dump(explainer, '../src/explainer.pkl')